# Homework Starter — Stage 6: Data Preprocessing
Name: Jesse Wang
Date: 2026-08-18

Objectives:
- Write modular cleaning functions in `src/cleaning.py`
- Load the raw dataset, apply the functions, save to `data/processed/`
- Compare original vs cleaned data and document all assumptions


In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    ("src/cleaning.py", "NEEDED", "YOU write this in the homework - the import fails until you do"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
else:
    print("\nAll needed files present.")

Looking in: /Users/wangjing/bootcamp_Jesse_Wang/homework/homework6

  [OK ]  NEEDED    src/cleaning.py                     YOU write this in the homework - the import fails until you do

All needed files present.


In [3]:
# --- generate the raw sample dataset (run me once) ---
import os
import pandas as pd
import numpy as np

raw_dir = 'data/raw'
processed_dir = 'data/processed'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
}
df = pd.DataFrame(data)

csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')

File already exists at data/raw/sample_data.csv. Skipping CSV creation to avoid overwrite.


## 1) Load Raw Dataset
The cleaning module lives in `src/cleaning.py` (imported below).

In [4]:
import pandas as pd
import numpy as np

from src import cleaning

df = pd.read_csv('data/raw/sample_data.csv')
print(df.shape)
df

(7, 6)


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## 2) Inspect Missingness
Check *where* the missing values are before choosing a strategy. This decides
whether to fill, drop rows, or drop columns.

In [5]:
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(1)

summary = pd.DataFrame({'missing': missing, 'pct': missing_pct})
summary.sort_values('missing', ascending=False)

,missing,pct
extra_data,5,71.4
income,3,42.9
age,1,14.3
score,1,14.3
zipcode,0,0.0
city,0,0.0


**Observation:** `extra_data` is missing in 5 of 7 rows (~71%). The other
columns are mostly complete. The missingness is **column-concentrated**, not
row-concentrated — so the right first move is to drop the near-empty column,
not to drop rows or impute a mostly-invented value.

Before the main pipeline, here is a quick behaviour check of the three modes of
`drop_missing` on the raw frame, to make the functions' semantics concrete.

In [6]:
# Behaviour check: the three drop_missing modes on the RAW frame
print('raw rows:', len(df))
print('drop_missing(columns=["income"]) ->',
      len(cleaning.drop_missing(df, columns=['income'])), 'rows (drops rows missing income)')
print('drop_missing(threshold=0.5)     ->',
      len(cleaning.drop_missing(df, threshold=0.5)), 'rows (keeps rows >=50% complete)')
print('drop_missing() [strict]          ->',
      len(cleaning.drop_missing(df)), 'rows (drops ANY row with ANY NaN)')

raw rows: 7
drop_missing(columns=["income"]) -> 4 rows (drops rows missing income)
drop_missing(threshold=0.5)     -> 7 rows (keeps rows >=50% complete)
drop_missing() [strict]          -> 0 rows (drops ANY row with ANY NaN)


## 3) Apply Cleaning Functions

In [7]:
# 3a. Drop the near-empty column first (documented assumption: non-essential, 71% missing)
df_clean = df.drop(columns=['extra_data'])

# 3b. Impute remaining numeric missing values with the column median
df_clean = cleaning.fill_missing_median(df_clean, columns=['age', 'income', 'score'])

# 3c. Guard: drop any row still >50% incomplete (none remain after 3a + 3b)
df_clean = cleaning.drop_missing(df_clean, threshold=0.5)

# 3d. Rescale numeric columns to [0, 1] so age/income/score are comparable
df_clean = cleaning.normalize_data(df_clean, columns=['age', 'income', 'score'], method='minmax')

df_clean

,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin
5,0.500000,0.6250,0.000000,12345,Unknown
6,0.571429,0.4375,0.538462,94105,San Francisco


## 4) Compare Original vs Cleaned

In [8]:
# Missing values: before vs after
before = df.isna().sum()
after = df_clean.isna().sum()
pd.DataFrame({'before': before, 'after': after}).fillna(0).astype(int)

,before,after
age,1,0
city,0,0
extra_data,5,0
income,3,0
score,1,0
zipcode,0,0


In [9]:
# Numeric summary: raw (left) vs cleaned+scaled (right)
print('--- raw (numeric) ---')
print(df[['age', 'income', 'score']].describe().round(3))
print()
print('--- cleaned (min-max scaled to [0, 1]) ---')
print(df_clean[['age', 'income', 'score']].describe().round(3))

--- raw (numeric) ---
          age     income  score
count   6.000      4.000  6.000
mean   39.500  51000.000  0.802
std     7.556   7071.068  0.093
min    29.000  42000.000  0.650
25%    35.000  47250.000  0.768
50%    39.500  52000.000  0.805
75%    44.000  55750.000  0.865
max    50.000  58000.000  0.910

--- cleaned (min-max scaled to [0, 1]) ---
         age  income  score
count  7.000   7.000  7.000
mean   0.500   0.589  0.585
std    0.328   0.314  0.326
min    0.000   0.000  0.000
25%    0.333   0.531  0.481
50%    0.500   0.625  0.596
75%    0.667   0.719  0.769
max    1.000   1.000  1.000


## 5) Save Cleaned Dataset

In [10]:
out_path = 'data/processed/sample_data_cleaned.csv'
df_clean.to_csv(out_path, index=False)
print('Saved ->', out_path)

# Reload to verify the saved file is what we expect
reloaded = pd.read_csv(out_path)
assert reloaded.shape == df_clean.shape, 'shape mismatch on reload'
assert not reloaded.isna().any().any(), 'cleaned data still contains NaN'
print('Reload OK:', reloaded.shape, '| no missing values:', not reloaded.isna().any().any())

Saved -> data/processed/sample_data_cleaned.csv
Reload OK: (7, 5) | no missing values: True


## 6) Assumptions & Tradeoffs

- **Drop `extra_data`** — assumes a ~71%-missing column is non-essential and
  that keeping it (impute a mostly-made-up value, or drop 5/7 rows) is worse
  than removing it.
- **Median imputation** (`fill_missing_median`) — assumes missingness is
  **MCAR/MAR**, not MNAR. Median (not mean) keeps the imputation robust to
  outliers/skew. Tradeoff: imputation shrinks variance and can mask a real
  signal if the missingness is systematic.
- **Row-drop threshold 0.5** (`drop_missing`) — after the column drop + fill
  this removes nothing, which is expected here because the missingness was
  column-concentrated. The step is kept as a guard for future, row-sparse data.
- **Min-max scaling** (`normalize_data`) — makes `age`, `income`, `score`
  comparable on `[0, 1]`. Tradeoff: min-max is sensitive to extreme outliers
  (they compress the rest of the range); z-score (`method='standard'`) is the
  alternative when outliers matter.

These choices are reproduced in `README.md` under "Cleaning strategy" and
"Assumptions".